# Day 1 — Linear Algebra and Parametric Models

**Workshop:** Mathematical Foundations of Modern AI

Companion notebook to the Day 1 lecture notes. Four parts:

1. A 2-layer MLP from scratch in NumPy — just the forward pass, no training. The point is to see that a neural network is literally `x @ W1.T + b1`, ReLU, `h @ W2.T + b2`.
2. The same MLP in PyTorch, trained on MNIST.
3. Visualizing the learned hidden representation with PCA.
4. A small autoencoder with a 2-dimensional latent space, and a walk along that latent space.

Runs on Colab (no GPU needed) or any local Python 3.10+ with `torch`, `torchvision`, `numpy`, `matplotlib`, `scikit-learn`.

---

## Part 1 — A 2-layer MLP from scratch in NumPy

We will build the model exactly as the lecture notes describe:
$$
\mathbf{h} = \mathrm{ReLU}(W_1 \mathbf{x} + \mathbf{b}_1), \qquad \mathbf{y} = W_2 \mathbf{h} + \mathbf{b}_2
$$
with input dimension 784 (a flattened 28×28 image), hidden dimension 256, output dimension 10.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

def init_mlp(d_in: int, d_hidden: int, d_out: int):
    """Initialize the weights and biases of a 2-layer MLP using He initialization."""
    W1 = rng.standard_normal((d_hidden, d_in)) * np.sqrt(2.0 / d_in)
    b1 = np.zeros(d_hidden)
    W2 = rng.standard_normal((d_out, d_hidden)) * np.sqrt(2.0 / d_hidden)
    b2 = np.zeros(d_out)
    return W1, b1, W2, b2

def forward(x: np.ndarray, W1, b1, W2, b2):
    """Forward pass for a batch x of shape (batch, d_in). Returns (logits, hidden)."""
    h = np.maximum(0, x @ W1.T + b1)   # ReLU
    y = h @ W2.T + b2                  # linear output
    return y, h

W1, b1, W2, b2 = init_mlp(d_in=784, d_hidden=256, d_out=10)
n_params = W1.size + b1.size + W2.size + b2.size
print(f"NumPy MLP parameters: {n_params:,}")
# Should print 203,530 — matches the count in the lecture notes.

In [ ]:
# Sanity check: feed a random batch and inspect every intermediate shape
batch = rng.standard_normal((32, 784))
y, h = forward(batch, W1, b1, W2, b2)
print(f"Input batch shape:        {batch.shape}")
print(f"Hidden activations shape: {h.shape}")
print(f"Output logits shape:      {y.shape}")
print(f"Output (first row, untrained): {y[0]}")

The forward pass is six lines of NumPy. The model right now is just numbers — random, untrained. To make it learn we need gradients, which is Day 2.

---

## Part 2 — The same MLP in PyTorch, trained on MNIST

PyTorch's `nn.Linear` holds the same `W` and `b` we wrote by hand. The only difference is that PyTorch tracks gradients for us. We train for three epochs and check test accuracy.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))   # flatten 28x28 to 784
])

train_ds = datasets.MNIST(root=".", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root=".", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256)

In [ ]:
class TwoLayerMLP(nn.Module):
    def __init__(self, d_in: int = 784, d_hidden: int = 256, d_out: int = 10):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = torch.relu(self.fc1(x))
        return self.fc2(h)

    def hidden(self, x: torch.Tensor) -> torch.Tensor:
        return torch.relu(self.fc1(x))

model = TwoLayerMLP().to(device)
print(model)
print(f"PyTorch MLP parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
import time

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

def evaluate() -> float:
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total   += y.numel()
    return correct / total

t0 = time.time()
for epoch in range(3):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    acc = evaluate()
    print(f"Epoch {epoch+1}: test accuracy = {acc:.4f}  (elapsed {time.time()-t0:.1f}s)")

You should see roughly 97% test accuracy after three epochs. On CPU this runs in under a minute. On a GPU, a few seconds.

---

## Part 3 — Visualizing the hidden representation with PCA

The hidden layer is a function from $\mathbb{R}^{784}$ to $\mathbb{R}^{256}$. After training, that representation should organize digits by similarity. We extract the 256-dimensional hidden activations for the test set, project them to 2D with PCA, and color by true class.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

model.eval()
features, labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        h = model.hidden(x.to(device)).cpu().numpy()
        features.append(h)
        labels.append(y.numpy())
features = np.concatenate(features)
labels   = np.concatenate(labels)

proj = PCA(n_components=2).fit_transform(features)

fig, ax = plt.subplots(figsize=(8, 7))
scatter = ax.scatter(proj[:, 0], proj[:, 1], c=labels, cmap="tab10", s=4, alpha=0.6)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Hidden activations (PCA-projected) — colored by true digit")
plt.colorbar(scatter, ax=ax, label="digit")
plt.tight_layout()
plt.show()

Same digits cluster together. Nothing in the training objective told the network to do this — the network was only asked to predict the right label. The clustering emerged because organizing similar inputs nearby is the most efficient way to assign labels.

This is what people mean when they say a network "learns a representation."

---

## Part 4 — A small autoencoder with a 2D latent space

We now compress MNIST images down to two numbers and back. The bottleneck forces the model to find what is essential about each digit. After training we can inspect the 2D latent space directly and walk along straight lines in it.

Architecture:
$$
\mathbf{z} = f_{\text{enc}}(\mathbf{x}) \in \mathbb{R}^2, \qquad \hat{\mathbf{x}} = f_{\text{dec}}(\mathbf{z}) \in \mathbb{R}^{784}
$$

Trained with MSE reconstruction loss.

In [ ]:
class TinyAutoencoder(nn.Module):
    def __init__(self, d_in: int = 784, d_hidden: int = 128, d_latent: int = 2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.ReLU(),
            nn.Linear(d_hidden, d_latent)
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, d_hidden), nn.ReLU(),
            nn.Linear(d_hidden, d_in), nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor):
        z = self.encoder(x)
        return self.decoder(z), z

ae = TinyAutoencoder().to(device)
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
recon = nn.MSELoss()

for epoch in range(5):
    ae.train()
    total = 0.0
    for x, _ in train_loader:
        x = x.to(device)
        x_hat, _ = ae(x)
        loss = recon(x_hat, x)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * x.size(0)
    print(f"Epoch {epoch+1}: mean MSE = {total/len(train_ds):.4f}")

In [ ]:
# Plot the 2D latent space, colored by true digit
ae.eval()
zs, ys = [], []
with torch.no_grad():
    for x, y in test_loader:
        _, z = ae(x.to(device))
        zs.append(z.cpu().numpy())
        ys.append(y.numpy())
zs = np.concatenate(zs)
ys = np.concatenate(ys)

fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(zs[:, 0], zs[:, 1], c=ys, cmap="tab10", s=4, alpha=0.7)
ax.set_xlabel("z1")
ax.set_ylabel("z2")
ax.set_title("Autoencoder latent space (test set, 2 dimensions)")
plt.colorbar(sc, ax=ax, label="digit")
plt.tight_layout()
plt.show()

In [ ]:
# Walk a straight line in the latent space from the cluster center of '0' to that of '1'
with torch.no_grad():
    z0 = torch.tensor(zs[ys == 0].mean(axis=0), device=device, dtype=torch.float32)
    z1 = torch.tensor(zs[ys == 1].mean(axis=0), device=device, dtype=torch.float32)
    n_steps = 10
    alphas = torch.linspace(0, 1, n_steps, device=device).unsqueeze(1)
    walk = (1 - alphas) * z0 + alphas * z1   # (n_steps, 2)
    imgs = ae.decoder(walk).cpu().numpy().reshape(n_steps, 28, 28)

fig, axes = plt.subplots(1, n_steps, figsize=(n_steps * 1.2, 1.4))
for ax, img in zip(axes, imgs):
    ax.imshow(img, cmap="gray")
    ax.axis("off")
plt.suptitle("Walking the latent space from '0' to '1'")
plt.show()

The digits morph smoothly into each other. Every intermediate point is a valid image because the decoder has learned a continuous map from the 2D latent space to image space.

This is the geometric picture that the rest of the workshop builds on:

- Day 3 will replace the deterministic encoder with a probabilistic one (variational autoencoder) and show that diffusion models are this picture in disguise.
- Day 4 will ask which architectures bake in which inductive biases — and why a CNN's latent space looks different from an MLP's.
- Day 5 will ask how to tell whether the geometry we have learned is actually generalizable.

But the structures are all here, in this notebook.